<a href="https://colab.research.google.com/github/corrielynnyuill-debug/AIProject_Stocks-CLY/blob/main/AIProject_Stocks_Part2_CLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Mount drive in Colab
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets
# Set file path
DATA_PATH = "/content/drive/MyDrive/AIProject/"

# load datasets to DataFrame
prices = pd.read_csv(DATA_PATH + 'historical_stock_prices.csv')
stocks = pd.read_csv(DATA_PATH + 'historical_stocks.csv')

# Basic EDA
print(prices.info())
print('-'*80)
print(stocks.info())
print('-'*80)
print(prices.describe(include='all'))
print('-'*80)
print(stocks.describe(include='all'))
print('-'*80)

# Check missing values
print(prices.isna().sum())
print('-'*80)
print(stocks.isna().sum())
print('-'*80)

# Check duplicates
print(prices.duplicated().sum())
print('-'*80)
print(stocks.duplicated().sum())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20973889 entries, 0 to 20973888
Data columns (total 8 columns):
 #   Column     Dtype  
---  ------     -----  
 0   ticker     object 
 1   open       float64
 2   close      float64
 3   adj_close  float64
 4   low        float64
 5   high       float64
 6   volume     int64  
 7   date       object 
dtypes: float64(5), int64(1), object(2)
memory usage: 1.3+ GB
None
--------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6460 entries, 0 to 6459
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ticker    6460 non-null   object
 1   exchange  6460 non-null   object
 2   name      6460 non-null   object
 3   sector    5020 non-null   object
 4   industry  5020 non-null   obje

In [ ]:
from numpy._core.defchararray import lower
import pandas as pd
from sklearn.impute import KNNImputer
from scipy.stats import zscore

# Convert to datetime
prices['date'] = pd.to_datetime(prices['date'], errors='coerce')
prices = prices.sort_values(['ticker', 'date'])

# Define the columns to interpolate
numeric_cols = ['open', 'high', 'low', 'close', 'adj_close', 'volume']

# Interpolate numeric columns per ticker using transform for index alignment
prices[numeric_cols] = prices.groupby('ticker')[numeric_cols].transform(
    lambda g: g.interpolate(method='linear').bfill().ffill()
)

# Encode categorical columns for KNN Imputation
stocks_encoded = pd.get_dummies(stocks[['sector', 'industry']], dummy_na=True)

imputer = KNNImputer(n_neighbors=5)
# Perform imputation and then create the DataFrame
stocks_imputed_array = imputer.fit_transform(stocks_encoded)
stocks_imputed = pd.DataFrame(stocks_imputed_array, columns=stocks_encoded.columns)

# Outlier Detection
prices['z_close'] = prices.groupby('ticker')['close'].transform(zscore)
prices['z_volume'] = prices.groupby('ticker')['volume'].transform(zscore)

# Flag outliers
outliers = prices[(prices['z_close'].abs() > 4) | (prices['z_volume'].abs() > 4)]
print(outliers)


# IQR Outliers
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < (Q1 - 1.5 * IQR)) | (series > (Q3 + 1.5 * IQR))

prices['close_outlier'] = prices.groupby('ticker')['close'].transform(iqr_outliers)
prices['volume_outlier'] = prices.groupby('ticker')['volume'].transform(iqr_outliers)

# Cap Outliers with Winsorization
prices['close'] = prices.groupby('ticker')['close'].transform(
    lambda s: s.clip(lower=s.quantile(0.01), upper=s.quantile(0.99))

)

# Remove impossible values
prices = prices[(prices['close'] > 0) & (prices['volume'] > 0)]
prices = prices[prices['volume'] >= 0]

# Remove ticker/date duplicate rows
prices = prices.drop_duplicates(subset=['ticker', 'date'])
print('-'*80)


/usr/local/lib/python3.12/dist-packages/pandas/core/groupby/generic.py:557: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = func(group, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/groupby/generic.py:557: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = func(group, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/groupby/generic.py:557: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = func(group, *args, **kwargs)


         ticker        open       close  adj_close         low        high  \
13766141      A   32.546494   31.473534  27.494957   28.612303   35.765381   
13766149      A   30.713520   28.880543  25.229753   28.478184   30.758226   
13766556      A   80.829758  113.733902  99.356789   80.472099  115.879829   
13766564      A  111.587982  101.573677  88.733742   93.705292  115.164520   
13766575      A  104.434906  108.726753  94.982590  100.143059  108.726753   
...         ...         ...         ...        ...         ...         ...   
10902449   ZYNE    6.270000    7.600000   7.600000    6.200000    8.430000   
10902482   ZYNE    5.850000    6.420000   6.420000    5.420000    6.690000   
10902803   ZYNE   11.050000    9.440000   9.440000    9.380000   11.280000   
10902811   ZYNE   10.100000    8.360000   8.360000    8.360000   10.147000   
10904820   ZYNE    7.950000    7.910000   7.910000    7.430000    8.130000   

            volume       date   z_close   z_volume  
13766141  

In [ ]:
# Feature Engineering
# Rolling Features
prices['rolling_7'] = prices.groupby('ticker')['close'].transform(lambda x: x.rolling(7).mean())
prices['rolling_30'] = prices.groupby('ticker')['close'].transform(lambda x: x.rolling(30).mean())

# Volatility
prices['volatility'] = prices.groupby('ticker')['close'].transform(lambda x: x.rolling(7).std())

# Technical Indicators
delta = prices.groupby('ticker')['close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

roll_up = gain.rolling(window=14).mean()
roll_down = loss.rolling(window=14).mean()

RS = roll_up / roll_down
prices['RSI'] = 100 - (100 / (1 + RS))

# Normalization /Standardization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ['open', 'high', 'low', 'close', 'adj_close', 'volume', 'rolling_7', 'rolling_30', 'volatility']
prices[cols_to_scale] = scaler.fit_transform(prices[cols_to_scale])


